In [1]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import warnings
import os 
warnings.filterwarnings("ignore")
import config
from utils import *
import math 
from tensorflow.keras.callbacks import TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint




2025-03-01 17:29:15.027079: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740846555.038883    5423 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740846555.042703    5423 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-01 17:29:15.054594: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


{'Cargo plane': 635, 'Helicopter': 70, 'Small car': 4290, 'Bus': 2155, 'Truck': 2746, 'Motorboat': 1069, 'Fishing vessel': 706, 'Dump truck': 1236, 'Excavator': 789, 'Building': 4689, 'Storage tank': 1469, 'Shipping container': 1523}


I0000 00:00:1740846557.912519    5423 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6717 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1070, pci bus id: 0000:02:00.0, compute capability: 6.1


In [ ]:
from keras_tuner import Hyperband

# Set up Hyperband
tuner = Hyperband(
    build_fcnn,
    objective='val_accuracy',
    max_epochs=80,
    factor=3,
    directory='hyperband_search',
    project_name='fcnn_tuning',
    executions_per_trial=1  # Disallow parallel execution
)

# Define callback for the search
early_stop_tuner = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

# Run the hyperparameter search
batch_size = 256
train_generator, valid_generator, sz_train, sz_val = train_val_split(batch_size=batch_size)
train_steps = math.ceil(sz_train / batch_size)
valid_steps = math.ceil(sz_val / batch_size)

# hyperband_checkpoint = HyperbandCheckpointCallback(tuner)

tuner.search(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=valid_generator,
    validation_steps=valid_steps,
    epochs=100,
    callbacks=[early_stop_tuner]
)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters found:")
for param, value in best_hps.values.items():
    print(f"{param}: {value}")




Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
3                 |3                 |num_layers
640               |640               |neurons_0
0.00064302        |0.00064302        |l1_0
0.00027892        |0.00027892        |l2_0
leaky_relu        |leaky_relu        |activation_0
glorot            |glorot            |init_0
True              |True              |batch_norm_0
0.2               |0.2               |dropout_0
896               |896               |neurons_1
0.0029718         |0.0029718         |l1_1
0.028267          |0.028267          |l2_1
elu               |elu               |activation_1
he                |he                |init_1
False             |False             |batch_norm_1
0.4               |0.4               |dropout_1
896               |896               |neurons_2
0.0017783         |0.0017783         |l1_2
0.0045054         |0.0045054         |l2_2
leaky_relu        |leaky_relu        |activation_2
he                |he       

In [ ]:
# Build the model with the best hyperparameters
best_model = tuner.hypermodel.build(best_hps)

# Define callbacks for the final model
model_checkpoint = ModelCheckpoint(
    'best_hyperband_model.keras',
    monitor='val_accuracy',
    verbose=1,
    save_best_only=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    verbose=1
)
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    verbose=1,
    restore_best_weights=True
)
callbacks = [model_checkpoint, reduce_lr, early_stop]

# If you have custom callbacks, add them here
# callbacks = callbacks + [TimingCallback(), HistorySaverCallback()]

# Train the best model
history = best_model.fit(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=valid_generator,
    validation_steps=valid_steps,
    epochs=100,  # You can train longer since we have early stopping
    callbacks=callbacks,
    verbose=1
)

# Best validation model
best_idx = int(np.argmax(history.history['val_accuracy']))
best_value = np.max(history.history['val_accuracy'])
print('Best validation model: epoch ' + str(best_idx+1), ' - val_accuracy ' + str(best_value))

# Save the best hyperparameters to a file for future reference
import json
with open('best_hyperparameters.json', 'w') as f:
    json.dump(best_hps.values, f, indent=2)